# BEM convergence experiments

This notebook documents the convergence experiments for the boundary element discretizations implemented in **ngsbem**. The numerical experiments themselves are defined in separate Python scripts. Their results are stored in the static reference table `bem_results_ref.csv`, which is read below.

The purpose of this notebook is therefore not to rerun the BEM experiments, but to explain the underlying model problems, the discretizations, and the quantities shown in the convergence plots. The plotting code and the reference data are kept unchanged.

Three experiments are considered:

* the Laplace Dirichlet-to-Neumann problem,
* the Laplace Neumann-to-Dirichlet problem,
* electromagnetic scattering by a sphere using the Maxwell boundary integral formulation and a Mie-series reference solution.

For every experiment, the polynomial order is fixed while the mesh is refined. The resulting error and computational time are studied as functions of the number of degrees of freedom.


## Experimental data and visualization

The scripts `Laplace_DtN_Convergence.py`, `Laplace_NtD_Convergence.py`, and `Maxwell_Mie_Convergence.py` contain the numerical experiments that generate the corresponding convergence data. The plots in this notebook are deliberately based on the statically stored reference file

`bem_results_ref.csv`.

This separates generation of the reference results from their documentation and visualization.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("results_ref/bem_results_ref.csv")
# df.tail()

In [ ]:
# Filter by type
df_neu = df[df["type"] == "neumann"]
df_dirichlet = df[df["type"] == "dirichlet_semi"]
df_mie = df[df["type"] == "mie"]

In [ ]:
def plot_timing_and_error(df, name="Dirichlet to Neumann", save_fig=False):
    """
    Erzeugt folgende Plots für ein gegebenes DataFrame:
    1) Zeit vs. Freiheitsgrade (ndof)
    2) Fehler vs. nominale Gitterweite (h)
    3) Zeit + Fehler kombiniert (zwei y-Achsen)
    4) Effizienzplot: Fehler vs. Zeit

    Parameter
    ---------
    df : DataFrame mit Spalten ["order", "h", "ndof", "time", "err"]
    name : str, Bezeichnung (z.B. "Dirichlet" oder "Neumann")
    save_fig : bool, ob Plots als PNG gespeichert werden sollen
    """

    # Globale Marker und Farben (ggf. bei dir schon definiert)
    colors = ["red", "blue", "green", "orange", "red"]
    markers = ["x-", "v-", "*-", "*-", "x-"]

    if name == "Dirichlet to Neumann":
        figure_name = "laplace_DtN"
    elif name == "Neumann to Dirichlet":
        figure_name = "laplace_NtD"
    elif name == "Mie":
        figure_name = "maxwell_mie"
    else:
        figure_name = name.strip().lower()

    # --- Plot 1: Zeit vs. ndof ---
    plt.figure(figsize=(8, 6))
    for order in range(len(df["order"].unique())+1):
        df_plot = df[df["order"] == order]
        if df_plot.empty:
            continue
        plt.loglog(df_plot["ndof"], df_plot["time"], markers[order],
                   label=f'Order {order}', color=colors[order])

    plt.xlabel("Degrees of freedom [n]")
    plt.ylabel("Time [s]")
    plt.title(f"Timings: {name} Problem")
    plt.legend()
    plt.grid(True, which="both", ls="--")
    if save_fig:
        plt.savefig(f"{figure_name}_timings.png", dpi=300)
    plt.show()

    # --- Plot 2: Fehler vs. h ---
    plt.figure(figsize=(8, 6))
    for order in range(len(df["order"].unique())+1):
        df_plot = df[df["order"] == order]
        if df_plot.empty:
            continue
        plt.loglog(df_plot["h"], df_plot["err"], markers[order],
        label=f'Order {order}', color=colors[order])
        # Referenzkurve nahe an numerischen Daten platzieren
        scale = df_plot["err"].iloc[0]
        if name == "Neumann to Dirichlet":
            reforder = order
        elif name == "Dirichlet to Neumann":
            reforder = order 
        elif name == "Mie":
            reforder = order + 1
        else:
            reforder = order
        plt.loglog(df_plot["h"], scale*(df_plot["h"]/df_plot["h"].iloc[0])**reforder,
        '--', label=f'h^{reforder}', alpha=0.5, color=colors[order])
        
    plt.xlabel("Nominal mesh size $h$")
    if name == "Neumann to Dirichlet":
        plt.ylabel(r"$L_2$ error $|| \nabla (m - m_{ref})||$")
    elif name == "Dirichlet to Neumann": 
        plt.ylabel(r"$L_2$ error $|| j - j_{ref}||$")
    elif name == "Mie":
        plt.ylabel(r"$L_2$ error $|| \vec j - \vec j_{ref}||$")
    else:
        plt.ylabel(r"$L_2$ error||$")

        
    plt.title(f"Accuracy: Error {name} Problem")
    plt.legend()
    plt.grid(True, which="both", ls="--")
    if save_fig:
        plt.savefig(f"{figure_name}_accuracy.png", dpi=300)
    plt.show()

    # --- Plot 3: Zeit + Fehler kombiniert ---
    fig, ax1 = plt.subplots(figsize=(8, 6))
    for order in range(len(df["order"].unique())+1):
        df_plot = df[df["order"] == order]
        if df_plot.empty:
            continue
        ax1.loglog(df_plot["ndof"], df_plot["time"], markers[order],
                   label=f'Time (order {order})', color=colors[order])

    ax1.set_xlabel("Degrees of freedom [n]")
    ax1.set_ylabel("Time [s]", color="black")
    ax1.grid(True, which="both", ls="--")

    ax2 = ax1.twinx()
    ax2.set_yscale("log")
    for order in range(len(df["order"].unique())+1):
        df_plot = df[df["order"] == order]
        if df_plot.empty:
            continue
        ax2.bar(df_plot["ndof"], df_plot["err"],
                width=df_plot["ndof"]*0.05, alpha=0.3,
                color=colors[order], label=f'Error (order {order})')

    ax2.set_ylabel(r"$L_2$ error", color="black")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

    plt.title(f"Timing & Error {name} Problem")
    if save_fig:
        plt.savefig(f"{figure_name}_timing_and_error.png", dpi=300)
    plt.show()

    # --- Plot 4: Effizienzplot (Fehler vs. Zeit) ---
    plt.figure(figsize=(8, 6))
    for order in range(len(df["order"].unique())+1):
        df_plot = df[df["order"] == order]
        if df_plot.empty:
            continue
        plt.loglog(df_plot["time"], df_plot["err"], markers[order],
                   label=f'Order {order}', color=colors[order])

    plt.xlabel("Time [s]")
    if name == "Neumann to Dirichlet":
        plt.ylabel(r"$L_2$ error $|| \nabla (m - m_{ref})||$")
    elif name == "Dirichlet to Neumann": 
        plt.ylabel(r"$L_2$ error $|| j - j_{ref}||$")
    elif name == "Mie":
        plt.ylabel(r"$L_2$ error $|| \vec j - \vec j_{ref}||$")
    else:
        plt.ylabel(r"$L_2$ error||$")
     
    plt.title(f"Efficiency: {name} Problem")
    plt.legend(loc="best")
    plt.grid(True, which="both", ls="--")
    if save_fig:
        plt.savefig(f"{figure_name}_efficiency.png", dpi=300)
    plt.show()


## Laplace Dirichlet-to-Neumann problem

Let $\Omega \subset \mathbb{R}^3$ be the unit ball with boundary $\Gamma = \partial\Omega$. The Dirichlet-to-Neumann experiment starts from prescribed Dirichlet data $ u\big|_\Gamma$ and computes the corresponding Neumann data $j = \partial_n \, u \Big|_\Gamma$.

The analytical function used in the experiment is

$$
u_{\mathrm{exa}}(x,y,z) = \frac{1}{\sqrt{(x-1)^2+(y-1)^2+(z-1)^2}}.
$$

Its singularity is located outside the unit ball, hence it provides a smooth harmonic solution in $\Omega$ and an analytical Neumann trace on $\Gamma$.

The Dirichlet trace is interpolated in the boundary of `H1` space of order $p$, while the unknown Neumann trace is discretized in `SurfaceL2` of order $p-1$. The boundary integral equation implemented by the experiment has the form

$$ 
V( j ) = \left(\frac{1}{2} \, M + K\right)u, 
$$

where $V$ denotes the Laplace single-layer operator, $K$ the double-layer operator, and $M$ the coupling between the discrete trace spaces.

The discrete Neumann trace $j_h$ is compared with the analytical normal derivative

$$ 
j_{\mathrm{exa}} = \nabla u_{\mathrm{exa}}\cdot n.
$$

The reported error is the surface $L^2$ error

$$ 
\mathrm{err} =
\left\|
j_{\mathrm{exa}}-j_h
\right\|_{L^2(\Gamma_h)}.
$$

The following plots show the computational time and error versus the number of degrees of freedom for the different polynomial orders.


In [ ]:
plot_timing_and_error(df_neu, name="Dirichlet to Neumann", save_fig=True)

## Laplace Neumann-to-Dirichlet problem

The second experiment considers the inverse trace mapping. Starting from the prescribed Neumann trace

$$
j_{\mathrm{exa}}
=
\nabla u_{\mathrm{exa}}\cdot n,
$$

the boundary integral formulation reconstructs the corresponding Dirichlet data of the harmonic function $u$.

The same analytical harmonic function is used, i.e.,

$$
u_{\mathrm{exa}}(x,y,z)
=
\frac{1}{\sqrt{(x-1)^2+(y-1)^2+(z-1)^2}}.
$$

The Neumann data $j$ is represented in a `SurfaceL2` space, whereas the unknown Dirichlet trace $m$ is represented in the boundary of `H1` space. The implemented equation is based on the hypersingular formulation,

$$
(D+S) \, m =\left(\frac{1}{2}M^T-K^T\right)j,
$$

where $D$ denotes the hypersingular Laplace operator. The rank-one stabilization $S$ removes the constant kernel of the hypersingular operator on a closed surface.

For a scalar trace $v$ on the surface, the surface $H^1$ seminorm is defined by

$$
|v|_{H^1(\Gamma_h)}
=
\|\nabla_{\Gamma_h}v\|_{L^2(\Gamma_h)}.
$$

It is a seminorm rather than a norm because it does not detect additive constants: $|v+c|_{H^1(\Gamma_h)}=|v|_{H^1(\Gamma_h)}$ for every constant $c$, and in particular every constant function has seminorm zero. This matches the constant kernel of the hypersingular operator. Thus, the seminorm measures the Dirichlet trace as an equivalence class modulo constants, while the stabilization $S$ selects one representative of that class.

The experiment measures the error in the surface $H^1$ seminorm, implemented through the tangential gradients,

$$
\mathrm{err}
=
\left\|
\nabla_\Gamma m_{\mathrm{exa},h}
-
\nabla_\Gamma m_h
\right\|_{L^2(\Gamma_h)}.
$$

Here $m_{\mathrm{exa},h}$ denotes the interpolation of the analytical Dirichlet trace into the discrete `H1` space. Consequently, the plot tests the convergence of the computed Dirichlet trace in the energy-relevant surface-gradient seminorm.

> **Expected order.** Although the discrete Neumann-to-Dirichlet solution is `H1`-conforming, the error is measured in the surface $H^1$ seminorm rather than in the $L^2$ norm. For polynomial degree $p$, the optimal convergence rate is therefore $O(h^p)$; the higher $O(h^{p+1})$ rate applies to the corresponding $L^2$ error.


In [ ]:
plot_timing_and_error(df_dirichlet, name="Neumann to Dirichlet", save_fig=True)

## Maxwell Mie scattering

The third experiment considers electromagnetic scattering by a sphere of radius $0.25$ at wave number $\kappa = 5$ ($\kappa$ is not an interior eigenvalue of the Lapacian for the sphere).

The incident electric field is the plane wave

$$
E_{\mathrm{inc}}(x,y,z)
=
\begin{pmatrix}
1\\
0\\
0
\end{pmatrix}
e^{\,\mathrm{i}\kappa z}.
$$

The unknown surface current is discretized in `HDivSurface`, which provides the tangential surface finite element space used by the Maxwell boundary element formulation. The implemented electric-field integral operator is assembled from Helmholtz single-layer contributions and has the discrete form corresponding to

$$
V_\kappa
=
\kappa V_\kappa^{\mathrm{vec}}
-
\frac{1}{\kappa}V_\kappa^{\mathrm{div}}.
$$

The resulting linear system is solved with GMRES. For the sphere, an analytical Mie-series solution is available and is used as the reference surface current $j_{\mathrm{Mie}}$.

The reported error is

$$
\mathrm{err}
=
\left\|
j_h-j_{\mathrm{Mie}}
\right\|_{L^2(\Gamma_h)}.
$$

Thus, this experiment verifies convergence of the complete Maxwell BEM discretization against an independent analytical reference solution.


In [ ]:
plot_timing_and_error(df_mie, name="Mie", save_fig=True)

## Interpretation of the convergence plots

For all three experiments, results are grouped by polynomial order. The accuracy plots display the discretization error against the nominal mesh size $h$, so their slopes represent the classical convergence orders with respect to $h$. Timing and efficiency plots retain degrees of freedom or runtime as their cost measure.

The **error plots** are used to assess whether refinement produces the expected asymptotic decrease of the discretization error. Higher polynomial orders should lead to steeper convergence curves once the asymptotic regime is reached.

The **timing plots** complement the approximation results by showing the computational effort required for the corresponding discretizations. They should be interpreted together with the error curves: the relevant comparison is not only the convergence quality with respect to $h$, but also how many degrees of freedom and how much computational time are required to reach a given accuracy.

Because this notebook reads the static reference results, its figures are reproducible independently of rerunning the comparatively expensive BEM solves. New reference data can be generated by the corresponding Python experiment scripts and stored in the reference table before updating the documentation.
